# Notebook 08 — StratLake Strategy Backtest Artifact Review

This notebook continues from Notebook 07 and can reattach to an **existing StratLake archive checkpoint** after `DRIVE_FOLDER_NAME` is configured:

```text
Session: stratlake_q1_feature_consumption
Archive: stratlake-session-stratlake_q1_feature_consumption
Archive root:
Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME / "stratlake-trade-engine" / "sessions" / "stratlake_q1_feature_consumption" / "archives"
```

The goal is to run a native StratLake strategy workflow, inspect native artifacts, plot native outputs when available, and optionally refresh the archive checkpoint.

This notebook is native-first:
- use `stratlake-run-strategy` for strategy execution;
- use generated StratLake configs and artifacts;
- avoid notebook-only signal or normalization logic except as diagnostic inspection.


## 1. Install package dependencies

In [ ]:
# Core dependencies and packages used by the notebook workflow.
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine


## 2. Mount Google Drive and set credentials

In [ ]:
from pathlib import Path
import os
import getpass

# Google Drive mount for archive/session persistence.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped or unavailable outside Colab:", repr(exc))



In [ ]:
import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value

alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

if not alpaca_api_key_id or not alpaca_api_secret_key:
    raise ValueError("Missing Alpaca API credentials.")

os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
os.environ["ALPACA_FEED"] = "iex"

print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set but not printed.")

## 3. Configure workspace, Drive roots, and research windows

These defaults follow the Notebook 04-07 workflow shape while keeping Drive paths user-configured. Replace `DRIVE_FOLDER_NAME` before any live Colab/Drive execution. Active work stays under `/content`; Google Drive is used only for session persistence, backups, archives, and handoff packs. Override the session IDs only when reconnecting to a specific prior run.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
from datetime import datetime, timezone

# Colab-safe workspace roots.
# In Colab, keep active work under /content and use Drive only for persistence/archive packs.
try:
    IN_COLAB
except NameError:
    IN_COLAB = Path("/content").exists()

WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME if IN_COLAB else WORKSPACE_ROOT / "drive" / DRIVE_FOLDER_NAME

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError("Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders.")

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"

FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

FINTECH_SESSION_NAME = "fintech_stratlake_input"
STRATLAKE_SESSION_NAME = "stratlake_q1_feature_consumption"

# Optional: set these to reconnect Notebook 08 to an existing Notebook 05/06/07 run.
FINTECH_SESSION_ID_OVERRIDE = ""
STRATLAKE_SESSION_ID_OVERRIDE = ""

# Q1 research target plus padded build/ingestion windows.
# The warmup window gives rolling/lagged feature builders data before Q1 so early-Q1 rows are less likely to be NaN.
ANALYSIS_START = "2026-01-02"
ANALYSIS_END = "2026-03-31"

BACKFILL_START = "2025-11-03"   # warmup before Q1
BACKFILL_END = "2026-04-15"     # buffer after Q1 for forward-return checks

FEATURE_BUILD_START = BACKFILL_START
FEATURE_BUILD_END = BACKFILL_END

BACKFILL_SYMBOLS = "AAPL,MSFT,NVDA,SPY,QQQ"

for path in [FINTECH_ROOT, STRATLAKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("FINTECH_ROOT:", FINTECH_ROOT)
print("STRATLAKE_ROOT:", STRATLAKE_ROOT)
print("ANALYSIS window:", ANALYSIS_START, "to", ANALYSIS_END)
print("Padded backfill/build window:", BACKFILL_START, "to", BACKFILL_END)


## 4. Initialize or attach the Fintech notebook session

This creates the local Fintech project/session folders when needed and discovers the active `FINTECH_SESSION_ID` from the generated session manifest. The notebook still treats Fintech curated daily bars as the explicit handoff into StratLake through `MARKETLAKE_ROOT`.

In [ ]:
from pathlib import Path
import json
import subprocess

RUN_FINTECH_INIT_PROJECT = True
FORCE_FINTECH_INIT_PROJECT = False

fintech_init_cmd = [
    "fintech-init-project",
    "--root", FINTECH_ROOT.as_posix(),
    "--session-name", FINTECH_SESSION_NAME,
    "--with-session",
    "--colab-profile",
]
if FORCE_FINTECH_INIT_PROJECT:
    fintech_init_cmd.append("--force")

print("Fintech init command:")
print(" ".join(fintech_init_cmd))

if RUN_FINTECH_INIT_PROJECT:
    result = subprocess.run(
        fintech_init_cmd,
        cwd=WORKSPACE_ROOT,
        text=True,
        capture_output=True,
    )
    print("\nSTDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"fintech-init-project failed with return code {result.returncode}")
else:
    print("Fintech project/session init skipped.")

# Discover the active Fintech session ID.
if FINTECH_SESSION_ID_OVERRIDE:
    FINTECH_SESSION_ID = FINTECH_SESSION_ID_OVERRIDE
    FINTECH_SESSION_MANIFEST = FINTECH_ROOT / "artifacts" / "sessions" / FINTECH_SESSION_ID / "session_manifest.json"
else:
    manifest_root = FINTECH_ROOT / "artifacts" / "sessions"
    manifest_candidates = sorted(
        manifest_root.glob("*/session_manifest.json") if manifest_root.exists() else [],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not manifest_candidates:
        raise FileNotFoundError(
            "No Fintech session_manifest.json was found under "
            f"{manifest_root.as_posix()}"
        )
    FINTECH_SESSION_MANIFEST = manifest_candidates[0]
    FINTECH_SESSION_ID = FINTECH_SESSION_MANIFEST.parent.name

MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "daily_bars"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_ROOT / "sessions" / FINTECH_SESSION_ID
FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSION_ROOT / "backups"
FINTECH_BACKUP_PACK_DIR = FINTECH_DRIVE_BACKUP_ROOT / FINTECH_ARCHIVE_ID

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )

for path in [MARKETLAKE_ROOT, DAILY_BARS_ROOT, FINTECH_DRIVE_SESSION_ROOT, FINTECH_DRIVE_BACKUP_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("FINTECH_SESSION_MANIFEST:", FINTECH_SESSION_MANIFEST.as_posix())
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT.as_posix())
print("DAILY_BARS_ROOT:", DAILY_BARS_ROOT.as_posix())
print("FINTECH_BACKUP_PACK_DIR:", FINTECH_BACKUP_PACK_DIR.as_posix())


## 5. Initialize or attach the StratLake notebook session

This creates the local StratLake app/session folders, writes notebook configs using the native `--notebook-configs` flag, and attaches the session to the Fintech `MARKETLAKE_ROOT`. The archive identity below matches the Notebook 07 checkpoint instance you provided.

In [ ]:
from pathlib import Path
import os
import subprocess

RUN_STRATLAKE_INIT_SESSION = True
FORCE_STRATLAKE_NOTEBOOK_CONFIGS = False

# Native strategy config.
NATIVE_STRATEGY_NAME = "momentum_v1"

stratlake_init_cmd = [
    "stratlake-init-session",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--project-name", STRATLAKE_SESSION_NAME,
    "--marketlake-root", MARKETLAKE_ROOT.as_posix(),
    "--drive-root", DRIVE_ROOT.as_posix(),
    "--enable-drive-persistence",
    "--notebook-configs",
]
if FORCE_STRATLAKE_NOTEBOOK_CONFIGS:
    stratlake_init_cmd.append("--force-notebook-configs")

print("StratLake init command:")
print(" ".join(stratlake_init_cmd))

if RUN_STRATLAKE_INIT_SESSION:
    result = subprocess.run(
        stratlake_init_cmd,
        cwd=WORKSPACE_ROOT,
        text=True,
        capture_output=True,
    )
    print("\nSTDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"stratlake-init-session failed with return code {result.returncode}")
else:
    print("StratLake init skipped.")

# Attach to the Notebook 07 archive/session instance by default.
STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID_OVERRIDE or STRATLAKE_SESSION_NAME
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"

STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_ROOT / "sessions" / STRATLAKE_SESSION_ID
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

FEATURES_DAILY_ROOT = STRATLAKE_ROOT / "data" / "curated" / "features_daily"
STRATEGIES_CONFIG = STRATLAKE_ROOT / "configs" / "strategies.yml"

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )

for path in [STRATLAKE_DRIVE_SESSION_ROOT, STRATLAKE_DRIVE_ARCHIVE_ROOT, FEATURES_DAILY_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["STRATLAKE_ROOT"] = STRATLAKE_ROOT.as_posix()
os.environ["MARKETLAKE_ROOT"] = MARKETLAKE_ROOT.as_posix()

print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("STRATLAKE_ARCHIVE_ID:", STRATLAKE_ARCHIVE_ID)
print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("STRATLAKE_DRIVE_SESSION_ROOT:", STRATLAKE_DRIVE_SESSION_ROOT.as_posix())
print("STRATLAKE_DRIVE_ARCHIVE_ROOT:", STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix())
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("FEATURES_DAILY_ROOT:", FEATURES_DAILY_ROOT.as_posix())
print("STRATEGIES_CONFIG:", STRATEGIES_CONFIG.as_posix())


## 6. Verify attached session paths and notebook configs

In [ ]:
import pandas as pd

session_path_checks = [
    ("FINTECH_ROOT", FINTECH_ROOT),
    ("FINTECH_SESSION_MANIFEST", FINTECH_SESSION_MANIFEST),
    ("MARKETLAKE_ROOT", MARKETLAKE_ROOT),
    ("DAILY_BARS_ROOT", DAILY_BARS_ROOT),
    ("FINTECH_DRIVE_SESSION_ROOT", FINTECH_DRIVE_SESSION_ROOT),
    ("FINTECH_BACKUP_PACK_DIR", FINTECH_BACKUP_PACK_DIR),
    ("STRATLAKE_ROOT", STRATLAKE_ROOT),
    ("STRATLAKE_CONFIGS", STRATLAKE_ROOT / "configs"),
    ("STRATLAKE_UNIVERSE_YML", STRATLAKE_ROOT / "configs" / "universe.yml"),
    ("STRATLAKE_PATHS_YML", STRATLAKE_ROOT / "configs" / "paths.yml"),
    ("STRATLAKE_STRATEGIES_YML", STRATEGIES_CONFIG),
    ("FEATURES_DAILY_ROOT", FEATURES_DAILY_ROOT),
    ("STRATLAKE_DRIVE_SESSION_ROOT", STRATLAKE_DRIVE_SESSION_ROOT),
    ("STRATLAKE_ARCHIVE_PACK_DIR", STRATLAKE_ARCHIVE_PACK_DIR),
]

session_check_rows = []
for label, path in session_path_checks:
    session_check_rows.append({
        "label": label,
        "path": path.as_posix(),
        "exists": path.exists(),
        "is_dir": path.is_dir() if path.exists() else False,
        "is_file": path.is_file() if path.exists() else False,
    })

session_path_checks_df = pd.DataFrame(session_check_rows)
display(session_path_checks_df)

missing_required = session_path_checks_df[
    session_path_checks_df["label"].isin([
        "FINTECH_ROOT",
        "MARKETLAKE_ROOT",
        "STRATLAKE_ROOT",
        "STRATLAKE_UNIVERSE_YML",
        "STRATLAKE_PATHS_YML",
        "STRATLAKE_STRATEGIES_YML",
        "FEATURES_DAILY_ROOT",
    ])
    & ~session_path_checks_df["exists"]
]

if missing_required.empty:
    print("Core session/config paths are present.")
else:
    print("Missing core paths. Review initialization output above.")
    display(missing_required)


## 7. Verify the Notebook 07 StratLake archive checkpoint

In [ ]:
print("Expected StratLake archive pack:")
print(STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("Exists:", STRATLAKE_ARCHIVE_PACK_DIR.exists())

if STRATLAKE_ARCHIVE_PACK_DIR.exists():
    print("\nArchive contents preview:")
    for p in sorted(STRATLAKE_ARCHIVE_PACK_DIR.rglob("*"))[:200]:
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"{kind}: {p.as_posix()}")
else:
    print("\nArchive pack was not found at the expected path.")
    print("Confirm Notebook 07 archive checkpoint was executed with archive creation enabled.")
    print("Notebook 08 can still run if local StratLake data/configs exist under STRATLAKE_ROOT.")


## 8. Restore the Notebook 07 StratLake archive checkpoint

Notebook 08 is intended to review the strategy/backtest artifacts created from the Notebook 07 feature-consumption session. Restore the archived StratLake feature/artifact/config state from Google Drive into the active `/content/stratlake-trade-engine-demo` workspace before running native strategy review commands.

This uses the native StratLake restore command rather than reimplementing file-copy logic in notebook code.


In [ ]:
from pathlib import Path
import os

# ---------------------------------------------------------------------
# Restore StratLake archive from Notebook 07
# ---------------------------------------------------------------------
# Native restore command pattern:
#
#   stratlake-session-archive-restore-bootstrap \
#     --archive-root <full archive pack directory> \
#     --target-root <workspace root> \
#     --validate-before-restore \
#     --inspect-before-restore \
#     --overwrite-policy overwrite_allowed
# ---------------------------------------------------------------------

RUN_STRATLAKE_ARCHIVE_RESTORE = False

STRATLAKE_ROOT = Path(STRATLAKE_ROOT)
if not STRATLAKE_ROOT.is_absolute():
    STRATLAKE_ROOT = (Path("/content") / STRATLAKE_ROOT).resolve()

STRATLAKE_ARCHIVE_PACK_DIR = Path(STRATLAKE_ARCHIVE_PACK_DIR)
if not STRATLAKE_ARCHIVE_PACK_DIR.is_absolute():
    STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_ARCHIVE_PACK_DIR.resolve()

print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("Archive pack exists:", STRATLAKE_ARCHIVE_PACK_DIR.exists())

if not STRATLAKE_ARCHIVE_PACK_DIR.exists():
    raise FileNotFoundError(
        f"Archive pack not found: {STRATLAKE_ARCHIVE_PACK_DIR.as_posix()}"
    )

# The restore CLI can use --target-root . if cwd is STRATLAKE_ROOT.
os.chdir(STRATLAKE_ROOT)
print("Current working directory:", Path.cwd().as_posix())

if RUN_STRATLAKE_ARCHIVE_RESTORE:
    !stratlake-session-archive-restore-bootstrap \
      --archive-root {STRATLAKE_ARCHIVE_PACK_DIR.as_posix()} \
      --target-root . \
      --validate-before-restore \
      --inspect-before-restore \
      --overwrite-policy overwrite_allowed
else:
    print("Manual restore is off by default in committed source. Set RUN_STRATLAKE_ARCHIVE_RESTORE=True to restore after reviewing the command preview.")
    print("Restore command preview:")
    print(f"""!stratlake-session-archive-restore-bootstrap \\
  --archive-root {STRATLAKE_ARCHIVE_PACK_DIR.as_posix()} \\
  --target-root . \\
  --validate-before-restore \\
  --inspect-before-restore \\
  --overwrite-policy overwrite_allowed""")

## 9. Verify restored StratLake configs, features, and artifacts

After restore, confirm that the native inputs needed for Notebook 08 are present. The strategy run should use restored native files rather than notebook-created fallback data.


In [ ]:
import pandas as pd

restored_required_paths = [
    ("configs/universe.yml", STRATLAKE_ROOT / "configs" / "universe.yml"),
    ("configs/paths.yml", STRATLAKE_ROOT / "configs" / "paths.yml"),
    ("configs/strategies.yml", STRATLAKE_ROOT / "configs" / "strategies.yml"),
    ("data/curated/features_daily", STRATLAKE_ROOT / "data" / "curated" / "features_daily"),
    ("artifacts", STRATLAKE_ROOT / "artifacts"),
]

restore_check_rows = []
for label, path in restored_required_paths:
    restore_check_rows.append({
        "label": label,
        "path": path.as_posix(),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
    })

restore_checks = pd.DataFrame(restore_check_rows)
display(restore_checks)

missing_restore_inputs = restore_checks.loc[~restore_checks["exists"], "label"].tolist()
if missing_restore_inputs:
    raise FileNotFoundError(
        "Restored StratLake archive is missing required Notebook 08 inputs: "
        + ", ".join(missing_restore_inputs)
    )

feature_files = sorted((STRATLAKE_ROOT / "data" / "curated" / "features_daily").rglob("*.parquet"))
artifact_files = sorted((STRATLAKE_ROOT / "artifacts").rglob("*")) if (STRATLAKE_ROOT / "artifacts").exists() else []

print("Feature parquet files:", len(feature_files))
if feature_files:
    print("First feature file:", feature_files[0].as_posix())

print("Artifact files/directories:", len(artifact_files))


## 10. Verify native StratLake workspace inputs

In [ ]:
from pathlib import Path
import pandas as pd

required_paths = [
    STRATLAKE_ROOT,
    MARKETLAKE_ROOT,
    STRATLAKE_ROOT / "configs",
    STRATLAKE_ROOT / "configs" / "universe.yml",
    STRATLAKE_ROOT / "configs" / "paths.yml",
    STRATLAKE_ROOT / "configs" / "strategies.yml",
    STRATLAKE_ROOT / "data" / "curated" / "features_daily",
]

check_rows = []
for p in required_paths:
    check_rows.append({
        "path": p.as_posix(),
        "exists": p.exists(),
        "is_dir": p.is_dir() if p.exists() else False,
        "is_file": p.is_file() if p.exists() else False,
    })

input_checks = pd.DataFrame(check_rows)
display(input_checks)

missing = input_checks[~input_checks["exists"]]
if not missing.empty:
    print("Missing required or expected paths:")
    display(missing)
else:
    print("All expected native strategy input paths exist.")


## 11. Inspect native strategy registry

In [ ]:
import yaml
from pathlib import Path
import pandas as pd

strategies_config_path = STRATLAKE_ROOT / "configs" / "strategies.yml"

if not strategies_config_path.exists():
    print("No strategies.yml found at:", strategies_config_path.as_posix())
else:
    print("strategies.yml:", strategies_config_path.as_posix())
    raw_text = strategies_config_path.read_text(encoding="utf-8")
    print(raw_text[:2000])

    try:
        strategies_doc = yaml.safe_load(raw_text) or {}
        if isinstance(strategies_doc, dict):
            if "strategies" in strategies_doc and isinstance(strategies_doc["strategies"], dict):
                rows = [{"strategy": k, **(v if isinstance(v, dict) else {"value": v})}
                        for k, v in strategies_doc["strategies"].items()]
            else:
                rows = [{"key": k, "value_type": type(v).__name__} for k, v in strategies_doc.items()]
            if rows:
                display(pd.DataFrame(rows))
    except Exception as exc:
        print("Could not parse strategies.yml as YAML:", repr(exc))


## 12. Run native StratLake strategy backtest

In [ ]:
import subprocess
from pathlib import Path
import os
import re
import pandas as pd

RUN_NATIVE_STRATEGY_BACKTEST = True

native_strategy_completed = False
native_strategy_stdout = ""
native_strategy_stderr = ""
native_strategy_returncode = None

# Run from STRATLAKE_ROOT because the CLI expects configs/strategies.yml as a repo-relative path.
if STRATLAKE_ROOT.exists():
    os.chdir(STRATLAKE_ROOT)

cmd = [
    "stratlake-run-strategy",
    "--strategies-config", "configs/strategies.yml",
    "--strategy", NATIVE_STRATEGY_NAME,
    "--start", ANALYSIS_START,
    "--end", ANALYSIS_END,
]

print("Native strategy command:")
print(" ".join(cmd))
print("Current working directory:", Path.cwd().as_posix())

if RUN_NATIVE_STRATEGY_BACKTEST:
    result = subprocess.run(
        cmd,
        cwd=STRATLAKE_ROOT,
        text=True,
        capture_output=True,
    )

    native_strategy_stdout = result.stdout or ""
    native_strategy_stderr = result.stderr or ""
    native_strategy_returncode = result.returncode
    native_strategy_completed = result.returncode == 0

    print("\nSTDOUT:")
    print(native_strategy_stdout)

    if native_strategy_stderr:
        print("\nSTDERR:")
        print(native_strategy_stderr)

    print("\nReturn code:", native_strategy_returncode)

    if not native_strategy_completed:
        raise RuntimeError("Native StratLake strategy command failed.")
else:
    print("Dry run only. Set RUN_NATIVE_STRATEGY_BACKTEST=True to execute.")


In [ ]:
import re
import pandas as pd

# ---------------------------------------------------------------------
# Parse native StratLake strategy CLI output into structured notebook rows
# ---------------------------------------------------------------------

def _extract(pattern: str, text: str, default=None, cast=None):
    match = re.search(pattern, text, flags=re.MULTILINE)
    if not match:
        return default

    value = match.group(1).strip()

    if cast is None:
        return value

    try:
        return cast(value)
    except Exception:
        return default


def _extract_percent(pattern: str, text: str, default=None):
    value = _extract(pattern, text, default=default, cast=float)
    if value is None:
        return default
    return value / 100.0


native_strategy_row = {
    "strategy": _extract(r"^strategy:\s*(.+)$", native_strategy_stdout),
    "run_id": _extract(r"^run_id:\s*(.+)$", native_strategy_stdout),
    "analysis_start": ANALYSIS_START,
    "analysis_end": ANALYSIS_END,
    "completed": native_strategy_completed,
    "returncode": native_strategy_returncode,
    "cumulative_return": _extract(r"^cumulative_return:\s*([-0-9.]+)", native_strategy_stdout, cast=float),
    "sharpe_ratio": _extract(r"^sharpe_ratio:\s*([-0-9.]+)", native_strategy_stdout, cast=float),
    "long_pct": _extract_percent(r"- long:\s*([-0-9.]+)%", native_strategy_stdout),
    "short_pct": _extract_percent(r"short:\s*([-0-9.]+)%", native_strategy_stdout),
    "flat_pct": _extract_percent(r"flat:\s*([-0-9.]+)%", native_strategy_stdout),
    "trades": _extract(r"- trades:\s*([0-9]+)", native_strategy_stdout, cast=int),
    "turnover": _extract(r"turnover:\s*([-0-9.]+)", native_strategy_stdout, cast=float),
    "avg_holding_bars": _extract(r"- avg holding:\s*([-0-9.]+)\s*bars", native_strategy_stdout, cast=float),
    "qa_status": _extract(r"- status:\s*(.+)$", native_strategy_stdout),
    "qa_rows": _extract(r"- rows:\s*([0-9]+)", native_strategy_stdout, cast=int),
    "qa_symbols": _extract(r"symbols:\s*([0-9]+)", native_strategy_stdout, cast=int),
    "benchmark_return": _extract_percent(r"- benchmark return:\s*([-+0-9.]+)%", native_strategy_stdout),
    "excess_return": _extract_percent(r"- excess return:\s*([-+0-9.]+)%", native_strategy_stdout),
    "correlation": _extract(r"- correlation:\s*([-+0-9.]+)", native_strategy_stdout, cast=float),
    "stderr_warning": native_strategy_stderr.strip() if native_strategy_stderr else "",
}

smoke_result = pd.DataFrame([native_strategy_row])

display(smoke_result)

native_run_id = native_strategy_row["run_id"]
print("Native strategy run_id:", native_run_id)
print("Native strategy QA status:", native_strategy_row["qa_status"])

## 13. Parse native strategy output into review rows

In [ ]:
import re
import pandas as pd

def parse_percent(text_value):
    if text_value is None:
        return None
    return float(str(text_value).replace("%", "").strip()) / 100.0

def parse_native_strategy_stdout(stdout: str):
    row = {
        "strategy": None,
        "run_id": None,
        "cumulative_return": None,
        "sharpe_ratio": None,
        "long_pct": None,
        "short_pct": None,
        "flat_pct": None,
        "trades": None,
        "turnover": None,
        "avg_holding_bars": None,
        "qa_status": None,
        "qa_rows": None,
        "qa_symbols": None,
        "benchmark_return": None,
        "excess_return": None,
        "correlation": None,
    }

    patterns = {
        "strategy": r"^strategy:\s*(.+)$",
        "run_id": r"^run_id:\s*(.+)$",
        "cumulative_return": r"^cumulative_return:\s*([-+0-9.eE]+)",
        "sharpe_ratio": r"^sharpe_ratio:\s*([-+0-9.eE]+)",
        "qa_status": r"^- status:\s*(.+)$",
    }

    for key, pattern in patterns.items():
        m = re.search(pattern, stdout, flags=re.MULTILINE)
        if m:
            value = m.group(1).strip()
            if key in {"cumulative_return", "sharpe_ratio"}:
                row[key] = float(value)
            else:
                row[key] = value

    m = re.search(r"- long:\s*([0-9.]+%)\s*\|\s*short:\s*([0-9.]+%)\s*\|\s*flat:\s*([0-9.]+%)", stdout)
    if m:
        row["long_pct"] = parse_percent(m.group(1))
        row["short_pct"] = parse_percent(m.group(2))
        row["flat_pct"] = parse_percent(m.group(3))

    m = re.search(r"- trades:\s*([0-9]+)\s*\|\s*turnover:\s*([-+0-9.eE]+)", stdout)
    if m:
        row["trades"] = int(m.group(1))
        row["turnover"] = float(m.group(2))

    m = re.search(r"- avg holding:\s*([-+0-9.eE]+)\s*bars", stdout)
    if m:
        row["avg_holding_bars"] = float(m.group(1))

    m = re.search(r"- rows:\s*([0-9]+)\s*\|\s*symbols:\s*([0-9]+)", stdout)
    if m:
        row["qa_rows"] = int(m.group(1))
        row["qa_symbols"] = int(m.group(2))

    m = re.search(r"- benchmark return:\s*([-+0-9.]+%)", stdout)
    if m:
        row["benchmark_return"] = parse_percent(m.group(1))

    m = re.search(r"- excess return:\s*([-+0-9.]+%)", stdout)
    if m:
        row["excess_return"] = parse_percent(m.group(1))

    m = re.search(r"- correlation:\s*([-+0-9.eE]+)", stdout)
    if m:
        row["correlation"] = float(m.group(1))

    return row

strategy_result_row = parse_native_strategy_stdout(native_strategy_stdout)
strategy_result_row["start"] = ANALYSIS_START
strategy_result_row["end"] = ANALYSIS_END
strategy_result_row["returncode"] = native_strategy_returncode
strategy_result_row["completed"] = native_strategy_completed

strategy_result = pd.DataFrame([strategy_result_row])
display(strategy_result)

run_id = strategy_result_row.get("run_id")
print("Parsed run_id:", run_id)


## 14. Discover native artifacts for the run

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

artifact_search_roots = [
    STRATLAKE_ROOT / "artifacts",
    STRATLAKE_ROOT / "reports",
    STRATLAKE_ROOT / "data",
]

artifact_rows = []
for root in artifact_search_roots:
    if not root.exists():
        continue

    for p in sorted(root.rglob("*")):
        if not p.is_file():
            continue

        include = False
        if run_id and run_id in p.as_posix():
            include = True
        elif p.suffix.lower() in {".json", ".csv", ".parquet", ".md", ".html"}:
            # Include candidate artifacts for review even if the run_id is not in the filename.
            include = True

        if not include:
            continue

        artifact_rows.append({
            "root": root.as_posix(),
            "path": p.as_posix(),
            "relative_path": p.relative_to(root).as_posix(),
            "suffix": p.suffix.lower(),
            "size_bytes": p.stat().st_size,
            "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).isoformat(),
            "matches_run_id": bool(run_id and run_id in p.as_posix()),
        })

artifact_inventory = (
    pd.DataFrame(artifact_rows)
    .drop_duplicates(subset=["path"])
    .sort_values(["matches_run_id", "modified_utc", "path"], ascending=[False, False, True])
    if artifact_rows else pd.DataFrame()
)

print("Discovered candidate native artifacts:", len(artifact_inventory))
if not artifact_inventory.empty:
    display(artifact_inventory.head(200))
else:
    print("No candidate native artifacts found under artifacts/, reports/, or data/.")


## 15. Load plottable native time series when available

In [ ]:
import pandas as pd
from pathlib import Path

native_time_series = pd.DataFrame()
time_series_source = None

if not artifact_inventory.empty:
    # Prefer run_id-matching files and likely returns/equity/portfolio output names.
    candidates = artifact_inventory.copy()
    name_score_terms = ["equity", "return", "returns", "curve", "portfolio", "daily"]
    candidates["name_score"] = candidates["path"].str.lower().apply(
        lambda s: sum(term in s for term in name_score_terms)
    )
    candidates = candidates[
        candidates["suffix"].isin([".parquet", ".csv"])
    ].sort_values(["matches_run_id", "name_score", "modified_utc"], ascending=[False, False, False])

    for _, row in candidates.iterrows():
        p = Path(row["path"])
        try:
            if p.suffix.lower() == ".parquet":
                df = pd.read_parquet(p)
            elif p.suffix.lower() == ".csv":
                df = pd.read_csv(p)
            else:
                continue

            if df.empty:
                continue

            # Accept native time series if it has a date-like column and at least one numeric column.
            date_cols = [c for c in df.columns if str(c).lower() in {"date", "timestamp", "datetime", "bar_date"}]
            numeric_cols = df.select_dtypes(include="number").columns.tolist()

            if date_cols and numeric_cols:
                native_time_series = df.copy()
                time_series_source = p.as_posix()
                print("Selected native time-series artifact:", time_series_source)
                print("Shape:", native_time_series.shape)
                display(native_time_series.head())
                break
        except Exception as exc:
            print("Could not inspect candidate:", p.as_posix(), "|", repr(exc))

if native_time_series.empty:
    print("No plottable native time-series artifact was found.")
    print("The next cell will plot parsed native summary metrics instead.")


## 16. Plot native strategy review output

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if not native_time_series.empty:
    plot_df = native_time_series.copy()

    date_col = next(
        c for c in plot_df.columns
        if str(c).lower() in {"date", "timestamp", "datetime", "bar_date"}
    )
    plot_df[date_col] = pd.to_datetime(plot_df[date_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[date_col]).sort_values(date_col)

    # Prefer existing cumulative/equity columns, then returns.
    lower_cols = {str(c).lower(): c for c in plot_df.columns}
    preferred_cols = [
        "equity_curve",
        "equity",
        "cumulative_return",
        "cum_return",
        "portfolio_value",
        "return",
        "returns",
        "daily_return",
    ]

    plot_col = None
    for name in preferred_cols:
        if name in lower_cols and pd.api.types.is_numeric_dtype(plot_df[lower_cols[name]]):
            plot_col = lower_cols[name]
            break

    if plot_col is None:
        numeric_cols = plot_df.select_dtypes(include="number").columns.tolist()
        plot_col = numeric_cols[0] if numeric_cols else None

    if plot_col is None:
        print("Native time-series artifact had no numeric column to plot.")
    else:
        y = pd.to_numeric(plot_df[plot_col], errors="coerce").fillna(0.0)
        if str(plot_col).lower() in {"return", "returns", "daily_return"}:
            y = (1.0 + y).cumprod() - 1.0
            ylabel = "Cumulative return"
            title_suffix = f"cumulative from {plot_col}"
        else:
            ylabel = plot_col
            title_suffix = plot_col

        ax = pd.DataFrame({"date": plot_df[date_col], "value": y}).set_index("date")["value"].plot(
            title=f"Native StratLake strategy output: {title_suffix}",
            figsize=(10, 4),
        )
        ax.set_xlabel("Date")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        plt.show()

elif not strategy_result.empty:
    metric_cols = [
        "cumulative_return",
        "sharpe_ratio",
        "benchmark_return",
        "excess_return",
        "correlation",
        "turnover",
    ]
    available = [c for c in metric_cols if c in strategy_result.columns and pd.notna(strategy_result.loc[0, c])]

    if available:
        plot_df = strategy_result[available].T.reset_index()
        plot_df.columns = ["metric", "value"]
        ax = plot_df.set_index("metric")["value"].plot(
            kind="bar",
            title="Native StratLake strategy summary metrics",
            figsize=(10, 4),
        )
        ax.set_xlabel("Metric")
        ax.set_ylabel("Value")
        ax.grid(True, alpha=0.3)
        plt.show()
        display(plot_df)
    else:
        print("No parsed native strategy metrics available to plot.")
else:
    print("No native strategy output available to plot.")


## 17. Benchmark comparison review

In [ ]:
benchmark_cols = [
    "strategy",
    "start",
    "end",
    "cumulative_return",
    "benchmark_return",
    "excess_return",
    "correlation",
    "sharpe_ratio",
    "trades",
    "turnover",
    "qa_status",
    "qa_rows",
    "qa_symbols",
]

available_cols = [c for c in benchmark_cols if c in strategy_result.columns]
benchmark_review = strategy_result[available_cols].copy()
display(benchmark_review)


## 18. Optional archive checkpoint refresh

In [ ]:
import subprocess
from pathlib import Path

RUN_STRATLAKE_ARCHIVE_CHECKPOINT = False

archive_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--archive-id", STRATLAKE_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("StratLake archive checkpoint command:")
print(" ".join(archive_cmd))

if RUN_STRATLAKE_ARCHIVE_CHECKPOINT:
    result = subprocess.run(
        archive_cmd,
        cwd=STRATLAKE_ROOT,
        text=True,
        capture_output=True,
    )

    print("\nSTDOUT:")
    print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"StratLake archive checkpoint failed with return code {result.returncode}"
        )

    print("\nStratLake archive checkpoint completed.")
else:
    print("\nDry run only. Set RUN_STRATLAKE_ARCHIVE_CHECKPOINT=True to create/update the archive.")


## 19. Final Notebook 08 handoff summary

In [ ]:
summary = {
    "fintech_session_id": FINTECH_SESSION_ID,
    "stratlake_session_id": STRATLAKE_SESSION_ID,
    "stratlake_archive_id": STRATLAKE_ARCHIVE_ID,
    "archive_pack_dir": STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
    "archive_pack_exists": STRATLAKE_ARCHIVE_PACK_DIR.exists(),
    "archive_restore_enabled": RUN_STRATLAKE_ARCHIVE_RESTORE if "RUN_STRATLAKE_ARCHIVE_RESTORE" in globals() else None,
    "strategy": NATIVE_STRATEGY_NAME,
    "analysis_start": ANALYSIS_START,
    "analysis_end": ANALYSIS_END,
    "native_strategy_completed": native_strategy_completed,
    "run_id": run_id,
    "artifact_candidates": int(len(artifact_inventory)) if "artifact_inventory" in globals() and not artifact_inventory.empty else 0,
    "native_time_series_rows": int(len(native_time_series)) if "native_time_series" in globals() else 0,
}

display(pd.DataFrame([summary]))

print("Notebook 08 complete.")
print("Next likely notebook: Notebook 09 — strategy comparison, walk-forward validation, or alpha evaluation.")
